# 🐧 Palmer Penguins Species Classification & Data Lifecycle Pipeline
**MSU AI Club Workshop 01 Template Repository (Notebook Alternative)**  
*Use this notebook for live demonstration and exploratory analysis during Workshop 01.*

---

## Project Overview & Data Lifecycle Framework
This project builds an end-to-end Machine Learning classification pipeline to predict Palmer Archipelago penguin species (`Adelie`, `Chinstrap`, `Gentoo`) based on biological measurements (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`).

### The 5 Data Lifecycle Stages in this Pipeline:
1. **Ingestion**: Loading raw Palmer Station Antarctica LTER dataset (`penguins.csv`).
2. **Cleaning & Imputation**: Handling missing physical measurements with median/mode imputation.
3. **Exploratory Data Analysis & Preprocessing**: Analyzing bill/flipper feature distributions.
4. **Model Training & Evaluation**: Training a Random Forest Classifier and computing Precision, Recall, and Confusion Matrices.
5. **Inference & Artifact Deployment**: Exporting `penguin_model.pkl` and running predictions CLI.


In [ ]:
import os
import sys
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import plotly.express as px

# Ensure cross-platform UTF-8 encoding
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# STAGE 1: DATA INGESTION & RAW DIAGNOSTIC
# [Data Lifecycle Hint]: Always check missingness rates and feature schemas before training!
DATA_PATH = "penguins.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "https://raw.githubusercontent.com/lowell-monis/palmer-penguins-ml-template/main/penguins.csv"

df = pd.read_csv(DATA_PATH)
print(f"[Stage 1] Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head())

print("
[Data Lifecycle Diagnostic] Missing values count by column:")
print(df.isnull().sum())


### Stage 2: Data Lifecycle Cleaning & Imputation
**[Data Lifecycle Hint]**: Never drop missing rows blindly (`df.dropna()`) without inspecting demographic loss. We apply **Median Imputation** for continuous numeric features (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`) and **Mode Imputation** for categorical attributes (`sex`).


In [ ]:
df_clean = df.copy()

numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)
        print(f"[Lifecycle Imputation] Imputed {col} with median: {median_val:.1f}")

if df_clean["sex"].isnull().sum() > 0:
    mode_sex = df_clean["sex"].mode()[0]
    df_clean["sex"] = df_clean["sex"].fillna(mode_sex)
    print(f"[Lifecycle Imputation] Imputed sex with mode: {mode_sex}")

print(f"
Total clean observations preserved: {len(df_clean)} (Missing values remaining: {df_clean.isnull().sum().sum()})")


### Stage 3: Exploratory Data Analysis (EDA)
**[Data Lifecycle Hint]**: Visualize feature relationships to verify class separability before model selection.


In [ ]:
fig = px.scatter(
    df_clean,
    x="bill_length_mm",
    y="bill_depth_mm",
    color="species",
    size="body_mass_g",
    hover_data=["island", "sex"],
    title="Palmer Penguins: Bill Length vs Bill Depth Scatter",
    labels={"bill_length_mm": "Bill Length (mm)", "bill_depth_mm": "Bill Depth (mm)"},
    color_discrete_map={"Adelie": "#08ffff", "Chinstrap": "#ff0055", "Gentoo": "#ffcc00"}
)
fig.update_layout(template="plotly_dark", font_family="Rubik")
fig.show()


### Stage 4: Model Training & Cohort Evaluation
**[Data Lifecycle Hint]**: Use stratified train/test splitting and evaluate per-class precision/recall to prevent small classes (Chinstrap) from being masked by overall accuracy.


In [ ]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = df_clean[features]
y = df_clean["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Overall Model Accuracy: {acc:.1%}
")
print("--- Per-Species Classification Report ---")
print(classification_report(y_test, y_pred))

with open("penguin_model.pkl", "wb") as f:
    pickle.dump(clf, f)
print("Saved model artifact to penguin_model.pkl")


### Stage 5: Inference & Live Prediction Test
**[Data Lifecycle Hint]**: Validate real-time predictions on unseen measurement inputs.


In [ ]:
sample_input = pd.DataFrame([{
    "bill_length_mm": 48.5,
    "bill_depth_mm": 15.0,
    "flipper_length_mm": 217.0,
    "body_mass_g": 5000.0
}])

pred_species = clf.predict(sample_input)[0]
probs = clf.predict_proba(sample_input)[0]

print(f"Predicted Species: {pred_species.upper()}")
for species_name, prob in zip(clf.classes_, probs):
    print(f"  * {species_name:10s}: {prob:.1%}")
